# Compressor Electrical Design with NeqSim

This notebook demonstrates the electrical design workflow for a gas compression train.
After running a process simulation, the electrical design system automatically sizes
motors, VFDs, cables, switchgear, and classifies hazardous areas.

**Equipment:** 2-stage gas compression with intercooling (5 → 30 → 80 bara)

**Standards covered:**
- IEC 60034-30-1 — Motor efficiency classes
- IEC 60502 / IEC 60364 — Cable sizing and derating
- IEC 61439 — Switchgear
- IEEE 519 — VFD harmonic limits
- IECEx — Hazardous area classification

In [ ]:
# Setup — load NeqSim
import subprocess, sys, pathlib, os

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    if ns is not None:
        ns = neqsim_classes(ns)
        NEQSIM_MODE = "devtools"
        print("NeqSim loaded via devtools")
    else:
        raise RuntimeError("devtools returned None")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip")

import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Import Java classes
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

print(f"Mode: {NEQSIM_MODE}")

## 1. Build and Run the Process Simulation

Create a 2-stage compression train with intercooling:
- Feed: Natural gas at 5 bara, 25 °C, 50 000 kg/hr
- Stage 1: 5 → 30 bara
- Intercooler: cool to 35 °C
- Stage 2: 30 → 80 bara

In [ ]:
# Create natural gas fluid
gas = SystemSrkEos(273.15 + 25.0, 5.0)
gas.addComponent("methane", 0.85)
gas.addComponent("ethane", 0.07)
gas.addComponent("propane", 0.03)
gas.addComponent("i-butane", 0.005)
gas.addComponent("n-butane", 0.005)
gas.addComponent("nitrogen", 0.02)
gas.addComponent("CO2", 0.02)
gas.setMixingRule("classic")

# Build process
process = ProcessSystem()

feed = Stream("Feed", gas)
feed.setFlowRate(50000.0, "kg/hr")
process.add(feed)

comp1 = Compressor("1st Stage Compressor", feed)
comp1.setOutletPressure(30.0)
process.add(comp1)

cooler = Cooler("Intercooler", comp1.getOutletStream())
cooler.setOutTemperature(273.15 + 35.0)
process.add(cooler)

comp2 = Compressor("2nd Stage Compressor", cooler.getOutletStream())
comp2.setOutletPressure(80.0)
process.add(comp2)

# Run the process simulation
process.run()

# Print process results
print("=== Process Simulation Results ===")
print(f"1st Stage power:  {comp1.getPower('kW'):.1f} kW")
print(f"1st Stage outlet: {comp1.getOutletStream().getTemperature() - 273.15:.1f} C,"
      f" {comp1.getOutletStream().getPressure():.1f} bara")
print(f"2nd Stage power:  {comp2.getPower('kW'):.1f} kW")
print(f"2nd Stage outlet: {comp2.getOutletStream().getTemperature() - 273.15:.1f} C,"
      f" {comp2.getOutletStream().getPressure():.1f} bara")

## 2. Run Electrical Designs

After the process simulation, call `runAllElectricalDesigns()` to
size motors, VFDs, cables, and switchgear for every driven equipment.

In [ ]:
# Run electrical designs for all equipment
process.runAllElectricalDesigns()

# Get electrical design for 1st stage compressor
ed1 = process.getEquipmentElectricalDesign("1st Stage Compressor")
ed2 = process.getEquipmentElectricalDesign("2nd Stage Compressor")

print("=== 1st Stage Compressor — Electrical Design ===")
print(f"  Shaft power:        {ed1.getShaftPowerKW():.1f} kW")
print(f"  Electrical input:   {ed1.getElectricalInputKW():.1f} kW")
print(f"  Apparent power:     {ed1.getApparentPowerKVA():.1f} kVA")
print(f"  Reactive power:     {ed1.getReactivePowerKVAR():.1f} kVAR")
print(f"  Power factor:       {ed1.getPowerFactor():.3f}")
print(f"  Full-load current:  {ed1.getFullLoadCurrentA():.1f} A")
print(f"  Total losses:       {ed1.getTotalElectricalLossesKW():.1f} kW")
print(f"  Rated voltage:      {ed1.getRatedVoltageV():.0f} V")

print(f"\n  Motor:")
motor1 = ed1.getMotor()
print(f"    Rated power:      {motor1.getRatedPowerKW():.0f} kW")
print(f"    Efficiency:       {motor1.getEfficiencyPercent():.1f}% ({motor1.getEfficiencyClass()})")
print(f"    Speed:            {motor1.getRatedSpeedRPM():.0f} RPM")
print(f"    Current:          {motor1.getRatedCurrentA():.1f} A")
print(f"    Frame:            {motor1.getFrameSize()}")

print(f"\n  Power Cable:")
cable1 = ed1.getPowerCable()
print(f"    Cross-section:    {cable1.getCrossSectionMM2():.0f} mm²")
print(f"    Ampacity:         {cable1.getAmpacityA():.0f} A")
print(f"    Voltage drop:     {cable1.getVoltageDropPercent():.2f}%")

print(f"\n  Switchgear:")
sg1 = ed1.getSwitchgear()
print(f"    Type:             {sg1.getSwitchgearType()}")
print(f"    Starter:          {sg1.getStarterType()}")
print(f"    CB rating:        {sg1.getCircuitBreakerRatingA():.0f} A")

## 3. Full JSON Report

The `toJson()` method produces a comprehensive report covering
all electrical components.

In [ ]:
# Print full JSON report for 1st stage
json_str = ed1.toJson()
report = json.loads(str(json_str))
print(json.dumps(report, indent=2))

## 4. Comparison Table — Both Stages

Side-by-side comparison of the two compressor electrical designs.

In [ ]:
# Build comparison table
params = [
    ("Shaft power (kW)", ed1.getShaftPowerKW(), ed2.getShaftPowerKW()),
    ("Electrical input (kW)", ed1.getElectricalInputKW(), ed2.getElectricalInputKW()),
    ("Apparent power (kVA)", ed1.getApparentPowerKVA(), ed2.getApparentPowerKVA()),
    ("Reactive power (kVAR)", ed1.getReactivePowerKVAR(), ed2.getReactivePowerKVAR()),
    ("Power factor", ed1.getPowerFactor(), ed2.getPowerFactor()),
    ("Full-load current (A)", ed1.getFullLoadCurrentA(), ed2.getFullLoadCurrentA()),
    ("Motor rated power (kW)", ed1.getMotor().getRatedPowerKW(), ed2.getMotor().getRatedPowerKW()),
    ("Motor efficiency (%)", ed1.getMotor().getEfficiencyPercent(), ed2.getMotor().getEfficiencyPercent()),
    ("Motor frame size", ed1.getMotor().getFrameSize(), ed2.getMotor().getFrameSize()),
    ("Cable size (mm²)", ed1.getPowerCable().getCrossSectionMM2(), ed2.getPowerCable().getCrossSectionMM2()),
    ("Voltage drop (%)", ed1.getPowerCable().getVoltageDropPercent(), ed2.getPowerCable().getVoltageDropPercent()),
    ("CB rating (A)", ed1.getSwitchgear().getCircuitBreakerRatingA(), ed2.getSwitchgear().getCircuitBreakerRatingA()),
    ("Starter type", ed1.getSwitchgear().getStarterType(), ed2.getSwitchgear().getStarterType()),
    ("Rated voltage (V)", ed1.getRatedVoltageV(), ed2.getRatedVoltageV()),
    ("Total losses (kW)", ed1.getTotalElectricalLossesKW(), ed2.getTotalElectricalLossesKW()),
]

print(f"{'Parameter':<28} {'1st Stage':>14} {'2nd Stage':>14}")
print("-" * 58)
for name, v1, v2 in params:
    if isinstance(v1, str):
        print(f"{name:<28} {v1:>14} {v2:>14}")
    else:
        print(f"{name:<28} {v1:>14.1f} {v2:>14.1f}")

## 5. Motor Part-Load Performance Curves

Motors have peak efficiency at ~75% load. The power factor drops
significantly below 50% load, increasing reactive power demand.

In [ ]:
# Motor part-load performance curves
motor = ed1.getMotor()
loads = np.linspace(0.1, 1.2, 50)
efficiencies = [motor.getEfficiencyAtLoad(float(L)) for L in loads]
power_factors = [motor.getPowerFactorAtLoad(float(L)) for L in loads]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Efficiency vs load
ax1.plot(loads * 100, efficiencies, "b-", lw=2)
ax1.axvline(75, color="gray", ls="--", alpha=0.5, label="Peak efficiency zone")
ax1.axvline(100, color="gray", ls="--", alpha=0.5)
ax1.fill_betweenx([min(efficiencies), max(efficiencies)], 75, 100,
                  color="green", alpha=0.1)
ax1.set_xlabel("Load (%)")
ax1.set_ylabel("Efficiency (%)")
ax1.set_title(f"{motor.getRatedPowerKW():.0f} kW Motor — Efficiency vs Load")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Power factor vs load
ax2.plot(loads * 100, power_factors, "r-", lw=2)
ax2.axhline(0.85, color="gray", ls="--", alpha=0.5, label="PF = 0.85")
ax2.set_xlabel("Load (%)")
ax2.set_ylabel("Power Factor (cos \u03c6)")
ax2.set_title(f"{motor.getRatedPowerKW():.0f} kW Motor — Power Factor vs Load")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()
print(f"Motor: {motor.getRatedPowerKW():.0f} kW, {motor.getEfficiencyClass()},"
      f" {motor.getPoles()}-pole, {motor.getFrameSize()} frame")

## 6. Power Triangle Visualization

The power triangle shows the relationship between active power $P$ (kW),
reactive power $Q$ (kVAR), and apparent power $S$ (kVA):

$$S = \frac{P}{\cos\varphi}, \quad Q = S \sin(\arccos(\cos\varphi))$$

In [ ]:
# Power triangle for both stages
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, ed, title in [(ax1, ed1, "1st Stage"), (ax2, ed2, "2nd Stage")]:
    P = ed.getElectricalInputKW()
    Q = ed.getReactivePowerKVAR()
    S = ed.getApparentPowerKVA()
    pf = ed.getPowerFactor()

    # Draw triangle
    ax.arrow(0, 0, P, 0, head_width=S*0.02, head_length=P*0.02,
             fc="blue", ec="blue", lw=2)
    ax.arrow(P, 0, 0, Q, head_width=P*0.02, head_length=Q*0.02,
             fc="red", ec="red", lw=2)
    ax.plot([0, P], [0, Q], "g-", lw=2)

    # Labels
    ax.text(P/2, -S*0.06, f"P = {P:.0f} kW", ha="center", fontsize=11, color="blue")
    ax.text(P + S*0.05, Q/2, f"Q = {Q:.0f} kVAR", ha="left", fontsize=11, color="red")
    ax.text(P*0.35, Q*0.55, f"S = {S:.0f} kVA", ha="center", fontsize=11,
            color="green", rotation=np.degrees(np.arctan2(Q, P)))

    # Angle arc
    theta = np.linspace(0, np.arccos(pf), 30)
    r = S * 0.15
    ax.plot(r * np.cos(theta), r * np.sin(theta), "k-", lw=1)
    ax.text(r * 1.1, r * 0.15, f"\u03c6 = {np.degrees(np.arccos(pf)):.1f}\u00b0\n"
            f"cos\u03c6 = {pf:.3f}", fontsize=9)

    ax.set_xlim(-S*0.1, P + S*0.25)
    ax.set_ylim(-S*0.12, Q + S*0.15)
    ax.set_aspect("equal")
    ax.set_title(f"{title} — Power Triangle")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Efficiency Chain

The total efficiency from electrical supply to shaft output:

$$\eta_{\text{total}} = \eta_{\text{motor}} \times \eta_{\text{VFD}}$$

$$P_{\text{loss}} = P_{\text{electrical}} - P_{\text{shaft}}$$

In [ ]:
# Efficiency chain bar chart
fig, ax = plt.subplots(figsize=(10, 5))

stages = ["1st Stage", "2nd Stage"]
shaft_p = [ed1.getShaftPowerKW(), ed2.getShaftPowerKW()]
elec_p = [ed1.getElectricalInputKW(), ed2.getElectricalInputKW()]
losses = [ed1.getTotalElectricalLossesKW(), ed2.getTotalElectricalLossesKW()]
motor_eff = [ed1.getMotor().getEfficiencyPercent(), ed2.getMotor().getEfficiencyPercent()]

x = np.arange(len(stages))
w = 0.3

ax.bar(x - w/2, shaft_p, w, label="Shaft Power (kW)", color="steelblue")
ax.bar(x + w/2, elec_p, w, label="Electrical Input (kW)", color="coral")

# Add losses annotation
for i in range(len(stages)):
    overall_eff = shaft_p[i] / elec_p[i] * 100 if elec_p[i] > 0 else 0
    ax.text(x[i], max(shaft_p[i], elec_p[i]) + 20,
            f"\u03b7 = {overall_eff:.1f}%\nLoss = {losses[i]:.0f} kW",
            ha="center", fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel("Power (kW)")
ax.set_title("Compressor Shaft vs Electrical Power")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"\nTotal shaft power:      {sum(shaft_p):.1f} kW")
print(f"Total electrical input:  {sum(elec_p):.1f} kW")
print(f"Total electrical losses: {sum(losses):.1f} kW")

## Summary

This notebook demonstrated:

1. **Process-to-electrical workflow** — run simulation first, then electrical design
2. **Motor sizing** — IEC standard power steps with efficiency class
3. **Cable and switchgear** — automatically sized from motor current
4. **Part-load performance** — efficiency and power factor curves
5. **Power triangle** — active, reactive, and apparent power relationships
6. **Efficiency chain** — losses from supply to shaft
7. **JSON reports** — comprehensive data export for each equipment

See also:
- `process_plant_load_list.ipynb` — plant-wide electrical load analysis
- `motor_vfd_analysis.ipynb` — VFD topology, harmonics, and speed control